# 📌 Extracción

In [ ]:
import urllib.request
import json
import pandas as pd

url = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json"

# Obtener los datos JSON directamente con urllib
with urllib.request.urlopen(url) as response:
    raw = response.read().decode('utf-8')
    data = json.loads(raw)

# Convertir a DataFrame
df_raw = pd.DataFrame(data)

print(f"Datos cargados correctamente: {df_raw.shape[0]} filas, {df_raw.shape[1]} columnas")
df_raw.head(3)

# 🔧 Transformación

In [ ]:
# Desestructurar columnas anidadas con json_normalize
df_customer  = pd.json_normalize(df_raw['customer'])
df_phone     = pd.json_normalize(df_raw['phone'])
df_internet  = pd.json_normalize(df_raw['internet'])
df_account   = pd.json_normalize(df_raw['account'])

# Concatenar todo en un único DataFrame
df = pd.concat(
    [df_raw[['customerID', 'Churn']], df_customer, df_phone, df_internet, df_account],
    axis=1
)

# --- Limpieza ---
# Convertir Churn a valor numérico 0/1 para cálculos
df['Churn_num'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Convertir TotalCharges a numérico (puede venir como string)
df['Charges.Total'] = pd.to_numeric(df['Charges.Total'], errors='coerce')
df['Charges.Monthly'] = pd.to_numeric(df['Charges.Monthly'], errors='coerce')

# Reemplazar valores nulos en TotalCharges con la mediana
median_total = df['Charges.Total'].median()
df['Charges.Total'] = df['Charges.Total'].fillna(median_total)

print("Shape final del DataFrame:", df.shape)
print("\nTipos de datos:")
print(df.dtypes)
print("\nValores nulos por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df.head(3)

# 📊 Carga y Análisis

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='muted')

# ── 1. Distribución de Churn con gráfico de pastel ──────────────────────────
churn_counts = df['Churn'].value_counts()
labels = ['No Evasor', 'Evasor']
colors = ['#4C72B0', '#DD8452']
explode = (0, 0.07)

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    churn_counts.values,
    labels=labels,
    autopct='%1.1f%%',
    colors=colors,
    explode=explode,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight('bold')
ax.set_title('Distribución de la Evasión de Clientes (Churn)', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

total = churn_counts.sum()
print(f"Total clientes : {total}")
print(f"Evasores       : {churn_counts.get('Yes', 0)}  ({churn_counts.get('Yes',0)/total*100:.1f}%)")
print(f"No evasores    : {churn_counts.get('No', 0)}  ({churn_counts.get('No',0)/total*100:.1f}%)")

In [ ]:
# ── 2. Distribución de cargo mensual por Churn — Histograma con KDE ─────────
fig, ax = plt.subplots(figsize=(8, 5))

for label, color in [('No', '#4C72B0'), ('Yes', '#DD8452')]:
    subset = df[df['Churn'] == label]['Charges.Monthly'].dropna()
    ax.hist(subset, bins=30, alpha=0.5, color=color, density=True,
            label='No Evasor' if label == 'No' else 'Evasor')
    subset.plot.kde(ax=ax, color=color, linewidth=2)

ax.set_title('Distribución del Cargo Mensual según Churn', fontsize=13)
ax.set_xlabel('Cargo Mensual (USD)')
ax.set_ylabel('Densidad')
ax.legend(title='Churn')
plt.tight_layout()
plt.show()

print(df.groupby('Churn')['Charges.Monthly'].describe().round(2))

In [ ]:
# ── 3. Cargo total por tipo de contrato — Boxplot ───────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

contract_order = df['Contract'].dropna().unique().tolist()
sns.boxplot(
    x='Contract',
    y='Charges.Total',
    hue='Churn',
    data=df,
    order=sorted(contract_order),
    palette={'No': '#4C72B0', 'Yes': '#DD8452'},
    ax=ax
)
ax.set_title('Cargo Total por Tipo de Contrato y Churn', fontsize=13)
ax.set_xlabel('Tipo de Contrato')
ax.set_ylabel('Cargo Total (USD)')
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, ['No Evasor', 'Evasor'], title='Churn')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Tasa de evasión por Servicio de Internet — Gráfico de barras horizontales ──
churn_internet = (
    df.groupby('InternetService')['Churn_num']
    .mean()
    .mul(100)
    .sort_values(ascending=True)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(
    churn_internet['InternetService'],
    churn_internet['Churn_num'],
    color=sns.color_palette('Set2', len(churn_internet)),
    edgecolor='white'
)
for bar in bars:
    ax.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f"{bar.get_width():.1f}%",
        va='center', fontsize=11
    )
ax.set_title('Tasa de Evasión (%) por Servicio de Internet', fontsize=13)
ax.set_xlabel('Tasa de Evasión (%)')
ax.set_ylabel('Servicio de Internet')
ax.set_xlim(0, churn_internet['Churn_num'].max() + 8)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. Tasa de evasión por Método de Pago — Gráfico de barras horizontales ──
churn_payment = (
    df.groupby('PaymentMethod')['Churn_num']
    .mean()
    .mul(100)
    .sort_values(ascending=True)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(
    churn_payment['PaymentMethod'],
    churn_payment['Churn_num'],
    color=sns.color_palette('Blues_d', len(churn_payment)),
    edgecolor='white'
)
for bar in bars:
    ax.text(
        bar.get_width() + 0.3,
        bar.get_y() + bar.get_height() / 2,
        f"{bar.get_width():.1f}%",
        va='center', fontsize=11
    )
ax.set_title('Tasa de Evasión (%) por Método de Pago', fontsize=13)
ax.set_xlabel('Tasa de Evasión (%)')
ax.set_ylabel('Método de Pago')
ax.set_xlim(0, churn_payment['Churn_num'].max() + 8)
plt.tight_layout()
plt.show()

# 📄 Informe Final

In [ ]:
# ── 6. Mapa de calor: correlación de variables numéricas con Churn ───────────
num_cols = df.select_dtypes(include='number').columns.tolist()
corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Mapa de Calor — Correlación entre Variables Numéricas', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7. Panel resumen: Servicio de Internet + Método de Pago (subplots) ───────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- Subgráfico izquierdo: Servicio de Internet --
internet_pivot = (
    df.groupby(['InternetService', 'Churn'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={'No': 'No Evasor', 'Yes': 'Evasor'})
)
internet_pivot.plot(
    kind='bar',
    ax=axes[0],
    color=['#4C72B0', '#DD8452'],
    edgecolor='white',
    width=0.6
)
axes[0].set_title('Evasores por Tipo de Servicio de Internet', fontsize=12)
axes[0].set_xlabel('Servicio de Internet')
axes[0].set_ylabel('Cantidad de Clientes')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Churn')

# -- Subgráfico derecho: Método de Pago --
payment_pivot = (
    df.groupby(['PaymentMethod', 'Churn'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={'No': 'No Evasor', 'Yes': 'Evasor'})
)
payment_pivot.plot(
    kind='bar',
    ax=axes[1],
    color=['#4C72B0', '#DD8452'],
    edgecolor='white',
    width=0.6
)
axes[1].set_title('Evasores por Método de Pago', fontsize=12)
axes[1].set_xlabel('Método de Pago')
axes[1].set_ylabel('Cantidad de Clientes')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Churn')

plt.suptitle('Panel Resumen — Factores Asociados al Churn', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 8. Resumen estadístico final por grupo Churn ────────────────────────────
summary = (
    df.groupby('Churn')[['Charges.Monthly', 'Charges.Total', 'tenure']]
    .agg(['mean', 'median', 'std'])
    .round(2)
)
summary.columns = ['_'.join(col) for col in summary.columns]
print("=" * 60)
print("RESUMEN ESTADÍSTICO POR GRUPO CHURN")
print("=" * 60)
print(summary.to_string())
print()

# Tasa de evasión por tipo de contrato
print("TASA DE EVASIÓN POR TIPO DE CONTRATO")
print("-" * 40)
tasa_contrato = (
    df.groupby('Contract')['Churn_num']
    .mean()
    .mul(100)
    .round(1)
    .rename('Tasa_Churn_%')
)
print(tasa_contrato.to_string())
print()

# Tasa de evasión por método de pago
print("TASA DE EVASIÓN POR MÉTODO DE PAGO")
print("-" * 40)
tasa_pago = (
    df.groupby('PaymentMethod')['Churn_num']
    .mean()
    .mul(100)
    .round(1)
    .rename('Tasa_Churn_%')
)
print(tasa_pago.to_string())